# RF Coach Vision su Colab (GPU T4)

Il **codice** arriva da GitHub, i **file pesanti** (pesi dei modelli e video) da Google Drive.

**Preparazione, una volta sola.** Su Google Drive crea la cartella **Il mio Drive/rf_coach_vision/** e dentro:

| Sottocartella | Cosa metterci |
|---|---|
| `models/` | `tennis_yolo11.pt`, `yolo11n-pose.pt` |
| `ckpts/` | `TrackNet_best.pt`, `InpaintNet_best.pt` (da `tracknet3/ckpts` sul PC) |
| `inputs/` | i video da analizzare |

`outputs/` e `pred_result/` vengono creati da solo alla fine, con i risultati.

**Ogni volta:** menu **Runtime → Cambia tipo di runtime → GPU T4**, poi esegui le celle dall'alto in basso.

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Scarica il codice da GitHub

Se il repository è **privato**, serve un token: pannello a sinistra, icona della chiave (Secrets),
aggiungi un segreto chiamato `GITHUB_TOKEN` con un token di GitHub (Settings → Developer settings →
Personal access tokens), attiva l'accesso al notebook, e la cella lo userà da sola.
Se lo rendi **pubblico**, il token non serve.

In [ ]:
import os, shutil, glob, subprocess

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], check=True)
print("codice scaricato:", sorted(os.listdir(WORK_DIR)))

## 4. Porta dal Drive modelli, video e risultati pallina già calcolati

In [ ]:
copie = [("models", ""), ("ckpts", "tracknet3/ckpts"), ("inputs", "inputs"), ("pred_result", "tracknet3/pred_result")]

for sub, dst in copie:
    src = os.path.join(DRIVE_DIR, sub)
    dst_dir = os.path.join(WORK_DIR, dst) if dst else WORK_DIR
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.isdir(src):
        for f in glob.glob(os.path.join(src, "*")):
            if os.path.isfile(f):
                shutil.copy(f, dst_dir)

mancanti = [f for f in ["tennis_yolo11.pt", "yolo11n-pose.pt", "tracknet3/ckpts/TrackNet_best.pt"]
            if not os.path.exists(os.path.join(WORK_DIR, f))]
assert not mancanti, f"Mancano su Drive: {mancanti} (vedi la tabella in cima al notebook)"

print("video disponibili:", sorted(os.listdir(os.path.join(WORK_DIR, "inputs"))))

## 5. Installa le librerie (PyTorch c'è già su Colab)

In [ ]:
!pip install -q ultralytics parse

## 6. Analisi

`TRACKNET_MODE`: `"weight"` (preciso) o `"nonoverlap"` (veloce).
`FORCE`: `True` ricalcola la pallina anche se è già in `pred_result`.

In [ ]:
VIDEO = "zverev_djokovic_trim_swin_like.mp4"
TRACKNET_MODE = "weight"
FORCE = False

import re, time
p = os.path.join(WORK_DIR, "analyze.py")
s = open(p).read()
s = re.sub(r"^TRACKNET_MODE = .*$", f'TRACKNET_MODE = "{TRACKNET_MODE}"', s, flags=re.M)
s = re.sub(r"^TRACKNET_FORCE_RECOMPUTE = .*$", f"TRACKNET_FORCE_RECOMPUTE = {FORCE}", s, flags=re.M)
open(p, "w").write(s)

%cd {WORK_DIR}
t0 = time.time()
!python analyze.py inputs/{VIDEO}
print(f"\nTempo totale: {(time.time() - t0) / 60:.1f} minuti")

## 7. Salva i risultati su Drive
Da eseguire sempre: lo spazio di lavoro di Colab sparisce alla chiusura della sessione.

In [ ]:
for sub, src in [("outputs", "outputs"), ("pred_result", "tracknet3/pred_result")]:
    dst = os.path.join(DRIVE_DIR, sub)
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(os.path.join(WORK_DIR, src, "*")):
        shutil.copy(f, dst)
        print("salvato:", os.path.join(dst, os.path.basename(f)))